# Airline Operations & Disruption Intelligence
## 03 — Data Cleaning

### Objective

The objective of this notebook is to clean the BTS flight dataset based
on the issues identified during the Data Quality Assessment.

The cleaning process will:

- Preserve valid flight records
- Correct data types where required
- Handle invalid or missing values appropriately
- Remove only confirmed duplicate records
- Standardize important fields
- Document every cleaning decision

### Important Principle

We will not remove data simply because it is unusual.

Every cleaning decision should have a clear business or data-quality reason.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path(r"D:\Data Analyst\EXCEL\api\flights_2026.csv")

flights = pd.read_csv(data_path)

print("Original shape:", flights.shape)

Original shape: (1847242, 44)


In [2]:
original_rows = len(flights)
original_columns = len(flights.columns)

print("Original rows:", original_rows)
print("Original columns:", original_columns)

Original rows: 1847242
Original columns: 44


## 1. Column Names

The BTS dataset already uses a consistent uppercase naming convention.

Therefore, column names will be retained as provided by the source.

No unnecessary renaming will be performed.

In [4]:
flights.columns.T

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'MKT_UNIQUE_CARRIER', 'BRANDED_CODE_SHARE', 'MKT_CARRIER_AIRLINE_ID',
       'MKT_CARRIER', 'MKT_CARRIER_FL_NUM', 'ORIGIN', 'ORIGIN_CITY_NAME',
       'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST', 'DEST_CITY_NAME',
       'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME',
       'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'TAXI_OUT', 'TAXI_IN',
       'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15',
       'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME',
       'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY',
       'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'],
      dtype='object')

In [ ]:
#view the sample data how it looks first 5.
flights.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,381.0,427.0,349.0,2475.0,0.0,0.0,43.0,0.0,0.0
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,321.0,292.0,270.0,2475.0,0.0,0.0,0.0,0.0,0.0
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,134.0,124.0,105.0,674.0,0.0,0.0,0.0,0.0,0.0
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,229.0,210.0,191.0,1709.0,0.0,0.0,0.0,0.0,0.0
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,153.0,147.0,117.0,728.0,0.0,0.0,0.0,0.0,0.0


In [26]:
#view the sample data how it looks last 5.
flights.tail()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1847237,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,80.0,85.0,54.0,402.0,0.0,0.0,5.0,0.0,16.0
1847238,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,140.0,143.0,126.0,1020.0,0.0,0.0,0.0,0.0,0.0
1847239,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,145.0,141.0,125.0,930.0,0.0,0.0,0.0,0.0,0.0
1847240,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,95.0,102.0,85.0,395.0,0.0,0.0,0.0,0.0,0.0
1847241,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,130.0,118.0,100.0,794.0,0.0,0.0,0.0,0.0,0.0


In [28]:
#view the sample data how it looks random 5.
flights.sample()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1125830,2026,1,2,26,4,2026-02-26,UA,UA_CODESHARE,19977,UA,...,0.0,101.0,108.0,66.0,403.0,0.0,0.0,0.0,0.0,0.0


In [29]:
#Describe is used to summerize the numeric values in table.
flights.describe()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER_FL_NUM,CRS_DEP_TIME,DEP_TIME,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
count,1847242.0,1847242.0,1.847242e+06,1.847242e+06,1.847242e+06,1847242,1.847242e+06,1.847242e+06,1.847242e+06,1.786458e+06,...,1.847242e+06,1.847241e+06,1.780185e+06,1.780185e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06,1.847242e+06
mean,2026.0,1.0,2.039273e+00,1.556970e+01,4.018039e+00,2026-02-15 16:33:46.758594560,1.982750e+04,2.764304e+03,1.322166e+03,1.329818e+03,...,2.564364e-03,1.480702e+02,1.415679e+02,1.135985e+02,8.080800e+02,5.571318e+00,1.106565e+00,3.017408e+00,2.713018e-02,6.091262e+00
min,2026.0,1.0,1.000000e+00,1.000000e+00,1.000000e+00,2026-01-01 00:00:00,1.939300e+04,1.000000e+00,1.000000e+00,1.000000e+00,...,0.000000e+00,-8.500000e+01,1.400000e+01,6.000000e+00,3.100000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2026.0,1.0,1.000000e+00,8.000000e+00,2.000000e+00,2026-01-24 00:00:00,1.979000e+04,1.332000e+03,9.080000e+02,9.130000e+02,...,0.000000e+00,9.500000e+01,8.900000e+01,6.200000e+01,3.730000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2026.0,1.0,2.000000e+00,1.600000e+01,4.000000e+00,2026-02-17 00:00:00,1.980500e+04,2.452000e+03,1.315000e+03,1.325000e+03,...,0.000000e+00,1.310000e+02,1.250000e+02,9.700000e+01,6.570000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2026.0,1.0,3.000000e+00,2.300000e+01,6.000000e+00,2026-03-11 00:00:00,1.997700e+04,4.145000e+03,1.730000e+03,1.739000e+03,...,0.000000e+00,1.800000e+02,1.730000e+02,1.440000e+02,1.050000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,2026.0,1.0,3.000000e+00,3.100000e+01,7.000000e+00,2026-03-31 00:00:00,2.043600e+04,9.914000e+03,2.359000e+03,2.400000e+03,...,1.000000e+00,1.202000e+03,7.630000e+02,7.190000e+02,4.983000e+03,3.339000e+03,1.958000e+03,1.560000e+03,1.600000e+03,2.338000e+03
std,0.0,0.0,8.309906e-01,8.701216e+00,2.017735e+00,NaN,2.667073e+02,1.724521e+03,4.851421e+02,5.011229e+02,...,5.057459e-02,7.225527e+01,7.243144e+01,7.074440e+01,5.896536e+02,3.898694e+01,1.901363e+01,1.699150e+01,1.953209e+00,3.190036e+01


## 2. Convert Flight Date

FL_DATE represents the date on which the flight operated.


In [5]:
flights["FL_DATE"] = pd.to_datetime(
    flights["FL_DATE"],
    format="mixed",
    errors="coerce"
)

print(flights["FL_DATE"].dtype)

datetime64[ns]


In [6]:
invalid_dates = flights["FL_DATE"].isna().sum()

print("Invalid/missing flight dates:", invalid_dates)

Invalid/missing flight dates: 0


## 3. Remove Exact Duplicate Records

In [7]:
duplicate_count = flights.duplicated().sum()

print("Exact duplicates:", duplicate_count)

Exact duplicates: 0


# Handling Missing Values

In [8]:
flights.sample(100)

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1381796,2026,1,3,10,2,2026-03-10,WN,WN,19393,WN,...,0.0,120.0,101.0,81.0,548.0,NaN,NaN,NaN,NaN,NaN
397493,2026,1,1,21,3,2026-01-21,AA,AA_CODESHARE,19805,AA,...,0.0,109.0,99.0,67.0,468.0,NaN,NaN,NaN,NaN,NaN
1785620,2026,1,3,29,7,2026-03-29,AA,AA_CODESHARE,19805,AA,...,0.0,151.0,137.0,100.0,742.0,NaN,NaN,NaN,NaN,NaN
1166228,2026,1,2,28,6,2026-02-28,UA,UA,19977,UA,...,0.0,302.0,274.0,255.0,1846.0,NaN,NaN,NaN,NaN,NaN
1234508,2026,1,3,3,2,2026-03-03,WN,WN,19393,WN,...,0.0,110.0,99.0,86.0,526.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1635039,2026,1,3,22,7,2026-03-22,DL,DL,19790,DL,...,0.0,114.0,98.0,79.0,534.0,NaN,NaN,NaN,NaN,NaN
635739,2026,1,2,2,1,2026-02-02,F9,F9,20436,F9,...,0.0,262.0,242.0,217.0,1476.0,NaN,NaN,NaN,NaN,NaN
1337472,2026,1,3,8,7,2026-03-08,UA,UA_CODESHARE,19977,UA,...,0.0,74.0,59.0,30.0,140.0,NaN,NaN,NaN,NaN,NaN
111826,2026,1,1,6,2,2026-01-06,AS,AS,19930,AS,...,0.0,165.0,162.0,128.0,987.0,NaN,NaN,NaN,NaN,NaN


In [9]:
flights.isna().sum().sort_values(ascending=False)

CANCELLATION_CODE         1784931
LATE_AIRCRAFT_DELAY       1462816
CARRIER_DELAY             1462816
SECURITY_DELAY            1462816
NAS_DELAY                 1462816
WEATHER_DELAY             1462816
AIR_TIME                    67057
ACTUAL_ELAPSED_TIME         67057
ARR_DELAY_NEW               67048
ARR_DELAY                   67048
ARR_DEL15                   67048
ARR_TIME                    62950
TAXI_IN                     62950
TAXI_OUT                    61991
DEP_DELAY                   60974
DEP_DEL15                   60974
DEP_DELAY_NEW               60974
DEP_TIME                    60784
CRS_ELAPSED_TIME                1
MONTH                           0
QUARTER                         0
YEAR                            0
BRANDED_CODE_SHARE              0
MKT_UNIQUE_CARRIER              0
FL_DATE                         0
DAY_OF_WEEK                     0
DAY_OF_MONTH                    0
MKT_CARRIER_FL_NUM              0
MKT_CARRIER                     0
MKT_CARRIER_AI

In [10]:
flights[flights['CANCELLATION_CODE']=="A"]

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
1303,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,180.0,NaN,NaN,986.0,NaN,NaN,NaN,NaN,NaN
1547,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,240.0,NaN,NaN,1460.0,NaN,NaN,NaN,NaN,NaN
2478,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0.0,139.0,NaN,NaN,813.0,NaN,NaN,NaN,NaN,NaN
5064,2026,1,1,1,4,2026-01-01,AA,AA_CODESHARE,19805,AA,...,0.0,123.0,NaN,NaN,562.0,NaN,NaN,NaN,NaN,NaN
5764,2026,1,1,1,4,2026-01-01,AS,AS,19930,AS,...,0.0,157.0,NaN,NaN,833.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1844566,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,65.0,NaN,NaN,239.0,NaN,NaN,NaN,NaN,NaN
1844991,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,145.0,NaN,NaN,895.0,NaN,NaN,NaN,NaN,NaN
1845075,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,75.0,NaN,NaN,239.0,NaN,NaN,NaN,NaN,NaN
1846157,2026,1,3,31,2,2026-03-31,WN,WN,19393,WN,...,0.0,280.0,NaN,NaN,1824.0,NaN,NaN,NaN,NaN,NaN


In [11]:
# --------------------------------------------------
# Handle delay-cause missing values
# --------------------------------------------------

delay_cause_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

flights[delay_cause_columns] = (
    flights[delay_cause_columns]
    .fillna(0)
)
delay_cause_columns

['CARRIER_DELAY',
 'WEATHER_DELAY',
 'NAS_DELAY',
 'SECURITY_DELAY',
 'LATE_AIRCRAFT_DELAY']

## 4. Cancellation Code

CANCELLATION_CODE is only applicable to cancelled flights.

Therefore, missing values for non-cancelled flights are considered
structurally valid and will be retained.

We will not replace these values simply to eliminate missingness.

In [12]:
delay_cause_columns = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

## 6. Delay Values

Negative departure and arrival delays can be valid.

For example:

- DEP_DELAY = -5 means the flight departed 5 minutes early.
- ARR_DELAY = -10 means the flight arrived 10 minutes early.

Therefore, negative delay values will NOT be removed.

In [13]:
print("Negative departure delays:")
print((flights["DEP_DELAY"] < 0).sum())

print("\nNegative arrival delays:")
print((flights["ARR_DELAY"] < 0).sum())

Negative departure delays:
1048051

Negative arrival delays:
1114558


In [14]:
invalid_distance = (
    flights["DISTANCE"] <= 0
).sum()

print("Invalid distance records:", invalid_distance)

Invalid distance records: 0


In [15]:
duration_columns = [
    "TAXI_OUT",
    "TAXI_IN",
    "CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME"
]

In [16]:
for column in duration_columns:
    invalid_count = (
        flights[column] < 0
    ).sum()

    print(f"{column}: {invalid_count}")

TAXI_OUT: 0
TAXI_IN: 0
CRS_ELAPSED_TIME: 8
ACTUAL_ELAPSED_TIME: 0
AIR_TIME: 0


In [17]:
binary_columns = [
    "CANCELLED",
    "DIVERTED",
    "DEP_DEL15",
    "ARR_DEL15"
]

In [18]:
for column in binary_columns:
    invalid = flights[
        ~flights[column].isin([0, 1]) &
        flights[column].notna()
    ]

    print(
        f"{column}: {len(invalid)} invalid values"
    )

CANCELLED: 0 invalid values
DIVERTED: 0 invalid values
DEP_DEL15: 0 invalid values
ARR_DEL15: 0 invalid values


In [19]:
print(
    "Missing origin:",
    flights["ORIGIN"].isna().sum()
)

print(
    "Missing destination:",
    flights["DEST"].isna().sum()
)

Missing origin: 0
Missing destination: 0


In [20]:
flights[
    flights["ORIGIN"].isna() |
    flights["DEST"].isna()
].head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY


In [21]:
print(
    "Missing airline codes:",
    flights["MKT_UNIQUE_CARRIER"].isna().sum()
)

Missing airline codes: 0


In [22]:
final_missing = (
    flights.isna()
    .sum()
    .sort_values(ascending=False)
)

final_missing[final_missing > 0]

CANCELLATION_CODE      1784931
ACTUAL_ELAPSED_TIME      67057
AIR_TIME                 67057
ARR_DELAY                67048
ARR_DELAY_NEW            67048
ARR_DEL15                67048
TAXI_IN                  62950
ARR_TIME                 62950
TAXI_OUT                 61991
DEP_DEL15                60974
DEP_DELAY                60974
DEP_DELAY_NEW            60974
DEP_TIME                 60784
CRS_ELAPSED_TIME             1
dtype: int64